# INOVAAPPS 2026 · Score de risco e fila de atendimento

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display

from dados import CORES, carregar, fmt_brl
from score import (FEATURES, VARIAVEIS, auc_por_mes, contribuicoes, precisao_topo, rodar)

pio.renderers.default = "plotly_mimetype+notebook_connected"
pio.templates["inova"] = go.layout.Template(layout=dict(
    font=dict(family="Inter, Segoe UI, system-ui, sans-serif", size=13, color="#2b2b29"),
    title=dict(font=dict(size=17), x=0.01, xanchor="left"),
    separators=",.", paper_bgcolor="#ffffff", plot_bgcolor="#ffffff",
    hoverlabel=dict(bgcolor="#ffffff", bordercolor="#c9c7c0", font=dict(size=12, color="#2b2b29")),
    legend=dict(orientation="h", yanchor="bottom", y=1.06, xanchor="right", x=1, title_text=""),
    margin=dict(l=70, r=30, t=110, b=60),
    xaxis=dict(gridcolor="#eceae5", linecolor="#bdbbb4", zeroline=False),
    yaxis=dict(gridcolor="#eceae5", linecolor="#bdbbb4", zeroline=False),
))
pio.templates.default = "plotly_white+inova"
CINZA, CINZA_CLARO, TINTA = "#8a8984", "#eeede9", "#2b2b29"
AMBAR = "#d68a00"
COR_FAIXA = {"alto": CORES["Cancelou"], "atencao": AMBAR, "baixo": CORES["Ativo"]}
NOME_FAIXA = {"alto": "Risco alto", "atencao": "Atenção", "baixo": "Baixo"}
COR_METODO = {"so_uso": "#b5b3ac", "regra": "#52514e", "logistica_alto": "#1c5cab", "logistica_alto_ou_atencao": "#7aa7dd"}
ROT = {f: v.rotulo for f, v in VARIAVEIS.items()}

def rgba(hex_, a):
    h = hex_.lstrip("#")
    return f"rgba({int(h[:2], 16)},{int(h[2:4], 16)},{int(h[4:], 16)},{a})"

def br(v, casas=2):
    return f"{v:.{casas}f}".replace(".", ",").replace("-", "−")

def pct(p, casas=1):
    return br(100 * p, casas) + "%"

def low(s):
    return s[0].lower() + s[1:]

def sub(txt):
    return f"<br><span style='font-size:12px;color:{CINZA}'>{txt}</span>"

d = carregar()
R = rodar(d)                       # painel → validação LOCO → cortes → treino final → fila
P, M, F, E, CUT = R.painel, R.modelo, R.fila, R.explicacao, R.cortes
T = P[P.treino]
N_CANC, N_ATIV = 22, 58
assert len(F) == N_ATIV and F.prioridade.tolist() == list(range(1, N_ATIV + 1))
print(f"Painel: {len(P)} linhas cliente × mês · {int(P.tem_dados.sum())} com dados · {len(T)} com histórico suficiente "
      f"para treino ({int(T.y.sum())} positivos = cancelou nos 3 meses seguintes; {int(P.tem_dados.sum()) - len(T)} linhas "
      f"com < 4 meses de dados ficam fora do treino) · C final = {br(M.C, 1)} · variáveis removidas por sinal invertido: "
      f"{', '.join(M.removidas) or 'nenhuma'}")

Painel: 1440 linhas cliente × mês · 1295 com dados · 1055 com histórico suficiente para treino (60 positivos = cancelou nos 3 meses seguintes; 240 linhas com < 4 meses de dados ficam fora do treino) · C final = 0,1 · variáveis removidas por sinal invertido: nenhuma


## 1. Peso de cada variável

In [2]:
co = M.coeficientes()
dob = R.dobras[M.features]
co["min"], co["max"] = dob.min(), dob.max()
co = co.reindex(co.coef.abs().sort_values().index)
cor = [CORES["Cancelou"] if c > 0 else CORES["Ativo"] for c in co.coef]
ylab = [f"<b>{r.rotulo}</b><br><span style='font-size:11px;color:{CINZA}'>{'↑ maior' if r.direcao > 0 else '↓ menor'} = mais risco</span>"
        for r in co.itertuples()]
unid = {"uso": "p.p.", "tempo": "h", "sla": "p.p.", "criticos": "/mês", "reclamacoes": "/mês", "atraso": "dias", "nps": "pontos",
        "nps_sem_nota": "", "persistencia": "meses"}
fig = go.Figure(go.Bar(
    x=co.coef, y=ylab, orientation="h", marker=dict(color=cor, line=dict(color="white", width=1)),
    error_x=dict(type="data", symmetric=False, array=co["max"] - co.coef, arrayminus=co.coef - co["min"], color=CINZA, thickness=1.2, width=4),
    text=[f"chances ×{br(np.exp(abs(c)))} por 1 desvio" for c in co.coef], textposition="outside", cliponaxis=False,
    customdata=np.c_[co.rotulo, co.desvio.map(lambda v: br(v, 2)), [unid[f] for f in co.index], co.odds_ratio.map(br),
                     co["min"].map(br), co["max"].map(br)],
    hovertemplate="<b>%{customdata[0]}</b><br>Coeficiente (padronizado): %{x:.2f}<br>1 desvio-padrão = %{customdata[1]} %{customdata[2]}"
                  "<br>Chances multiplicadas por %{customdata[3]} a cada +1 desvio"
                  "<br>Faixa nas 80 dobras da validação: %{customdata[4]} a %{customdata[5]}<extra></extra>"))
fig.add_vline(x=0, line=dict(color=TINTA, width=1))
top = co.coef.abs().sort_values(ascending=False).index
fig.update_xaxes(title_text="Coeficiente da regressão logística (log-odds por +1 desvio-padrão)", range=[co["min"].min() - 0.35, co["max"].max() + 0.45])
fig.update_layout(height=560, margin=dict(l=260, t=130), showlegend=False,
                  title=f"O que mais pesa no risco: {low(ROT[top[0]])}, {low(ROT[top[1]])} e {low(ROT[top[2]])}"
                        + sub(f"Regressão logística L2 (C = {br(M.C, 1)}) · variáveis padronizadas · vermelho: valor alto aumenta o risco · "
                              "azul: valor baixo aumenta o risco · traço = mín–máx nas 80 dobras (todas com o sinal esperado)"))
fig.show()
display(co[["rotulo", "coef", "odds_ratio", "media", "desvio", "min", "max", "sinal_ok"]].iloc[::-1].round(3)
        .rename(columns={"rotulo": "Variável", "coef": "Coeficiente", "odds_ratio": "Chances ×", "media": "Média", "desvio": "Desvio",
                         "min": "Mín (dobras)", "max": "Máx (dobras)", "sinal_ok": "Sinal esperado"}).rename_axis(None))

,Variável,Coeficiente,Chances ×,Média,Desvio,Mín (dobras),Máx (dobras),Sinal esperado
uso,Queda do uso vs média anterior,-0.636,0.529,-1.327,8.210,-0.771,-0.450,True
nps,Última nota NPS,-0.549,0.578,7.109,1.966,-0.654,-0.385,True
tempo,Tempo médio de resolução (3m),0.464,1.590,22.387,4.783,0.309,0.576,True
criticos,Chamados críticos (3m),0.451,1.570,0.777,0.693,0.310,0.507,True
sla,SLA cumprido (3m),-0.419,0.657,77.642,14.874,-0.486,-0.275,True
reclamacoes,Reclamações formais (3m),0.249,1.283,0.393,0.410,0.194,0.330,True
atraso,Atraso de pagamento (3m),0.108,1.114,3.136,2.635,0.065,0.170,True
persistencia,Meses seguidos em alerta,0.099,1.105,0.903,1.703,0.057,0.206,True
nps_sem_nota,Nunca respondeu o NPS,0.011,1.011,0.025,0.155,0.005,0.062,True


## 2. Validação sem vazamento × regras simples

In [3]:
BL = R.baselines
NOME = {"so_uso": "Só queda de uso (≥ 10 p.p.)", "regra": "Regra combinada (notebook 02)",
        "logistica_alto": f"Score: faixa alto (p ≥ {br(CUT['alto'])})",
        "logistica_alto_ou_atencao": f"Score: alto ou atenção (p ≥ {br(CUT['atencao'])})"}
itens = [("canc_k1", "Cancelados detectados<br>em −1", N_CANC), ("canc_k2", "… em −2", N_CANC), ("canc_k3", "… em −3", N_CANC),
         ("ativos_hoje", "Ativos em alarme<br>hoje (jun/2026)", N_ATIV), ("ativos_algum_mes", "Ativos com alarme em<br>algum mês (12 meses)", N_ATIV)]
fig = go.Figure()
for m, r in BL.items():
    fig.add_trace(go.Bar(x=[i[1] for i in itens], y=[100 * r[i[0]] / i[2] for i in itens], name=NOME[m], marker_color=COR_METODO[m],
                         marker_line=dict(color="white", width=1), text=[f"{int(r[i[0]])}" for i in itens], textposition="outside", cliponaxis=False,
                         customdata=[[int(r[i[0]]), i[2]] for i in itens],
                         hovertemplate=f"<b>{NOME[m]}</b><br>%{{x}}: %{{customdata[0]}} de %{{customdata[1]}} (%{{y:.0f}}%)<extra></extra>"))
fig.add_vrect(x0=2.5, x1=4.5, fillcolor=CINZA_CLARO, opacity=0.5, line_width=0, layer="below")
fig.add_annotation(x=3.5, y=108, text="alarme falso (menor é melhor)", showarrow=False, font=dict(size=11, color=CINZA))
fig.add_annotation(x=1, y=108, text="detecção (maior é melhor)", showarrow=False, font=dict(size=11, color=CINZA))
r, g = BL["logistica_alto_ou_atencao"], BL["regra"]
fig.update_yaxes(range=[0, 112], ticksuffix="%", title_text="% do grupo (cancelados: 22 · ativos: 58)")
fig.update_layout(barmode="group", bargap=0.2, bargroupgap=0.05, height=500, legend=dict(y=1.08),
                  title=f"Score (alto + atenção) × regra: {int(r['canc_k2'])} × {int(g['canc_k2'])} cancelados pegos 2 meses antes, "
                        f"{int(r['ativos_algum_mes'])} × {int(g['ativos_algum_mes'])} ativos alarmados no ano"
                        + sub("Probabilidades out-of-fold: cada cliente é avaliado por um modelo treinado sem ele (80 modelos) · "
                              "rótulo = nº de clientes"))
fig.show()

scores = {"Score (logística, out-of-fold)": (P.p_oof, 1), "Só queda de uso (variação contínua)": (P.uso, -1),
          "Nº de sinais disparados (regra)": (P.n_sinais, 1)}
AUC = auc_por_mes(P, scores)
COR_AUC = dict(zip(scores, [COR_METODO["logistica_alto"], COR_METODO["so_uso"], COR_METODO["regra"]]))
fig = go.Figure()
fig.add_hline(y=0.5, line=dict(color=CINZA, dash="dot", width=1.5))
for nome in scores:
    q = AUC[AUC.score == nome]
    fig.add_trace(go.Scatter(x=-q.k, y=q.auc, mode="lines+markers", name=nome, line=dict(color=COR_AUC[nome], width=2.5),
                             marker=dict(size=9, line=dict(color="white", width=1.5)), customdata=np.c_[q.k, q.n_canc, q.n - q.n_canc],
                             hovertemplate=f"<b>{nome}</b><br>Avaliado no mês −%{{customdata[0]}}<br>AUC %{{y:.3f}}"
                                           "<br>%{customdata[1]} cancelados · %{customdata[2]} ativos<extra></extra>"))
a = AUC.pivot(index="k", columns="score", values="auc")
fig.update_xaxes(tickvals=list(range(-6, 0)), ticktext=[f"−{k}" for k in range(6, 0, -1)], title_text="Mês da avaliação (meses antes da saída)")
fig.update_yaxes(range=[0.45, 1.01], tickformat=".2f", title_text="AUC Cancelou × Ativo")
lg, us = a["Score (logística, out-of-fold)"], a["Só queda de uso (variação contínua)"]
fig.update_layout(height=460, legend=dict(y=1.08),
                  title=f"Separação mês a mês: o score ordena melhor em −1 e −2 (AUC {br(lg[1])} e {br(lg[2])}); "
                        f"em −3 a queda de uso sozinha ainda separa mais ({br(us[3])} × {br(lg[3])})"
                        + sub("Um valor por cliente no mês −k · cancelados no próprio −k, ativos em −k a partir de jul/2026 · 0,50 = acaso"))
fig.show()

top = {"Score (logística)": precisao_topo(P, P.p_oof), "Só queda de uso": precisao_topo(P, P.uso, -1)}
tab = []
for m, r in BL.items():
    tab.append({"Método": NOME[m], **{f"Cancelados em −{k}": f"{int(r[f'canc_k{k}'])}/22" for k in (1, 2, 3)},
                "Antecedência mediana (meses)": br(r["antecedencia_mediana"], 1), "Ativos em alarme hoje": f"{int(r['ativos_hoje'])}/58",
                "Ativos c/ alarme em algum mês (12m)": f"{int(r['ativos_algum_mes'])}/58",
                "Meses em alarme nos ativos (12m)": br(r["meses_alarme_ativos_pct"], 1) + "%"})
display(pd.DataFrame(tab).set_index("Método"))
display(pd.concat({m: t.pivot(index="k", columns="top", values="cancelados") for m, t in top.items()}, axis=1)
        .rename(index=lambda k: f"Ranking no mês −{k}").rename_axis(index=None, columns=["Cancelados entre os primeiros", "Top"]))

,Cancelados em −1,Cancelados em −2,Cancelados em −3,Antecedência mediana (meses),Ativos em alarme hoje,Ativos c/ alarme em algum mês (12m),Meses em alarme nos ativos (12m)
Método,,,,,,,
Só queda de uso (≥ 10 p.p.),17/22,14/22,10/22,"3,0",1/58,23/58,"8,0%"
Regra combinada (notebook 02),22/22,17/22,14/22,"3,0",5/58,36/58,"16,4%"
"Score: faixa alto (p ≥ 0,20)",21/22,10/22,8/22,"1,0",2/58,13/58,"4,0%"
"Score: alto ou atenção (p ≥ 0,05)",21/22,20/22,14/22,"3,0",6/58,28/58,"14,5%"


Cancelados entre os primeiros Score (logística)     Só queda de uso    
Top                                          10  20              10  20
Ranking no mês −1                            10  19              10  18
Ranking no mês −2                             9  17              10  16
Ranking no mês −3                             8  14               8  13

## 3. Cortes das faixas

In [4]:
cv = R.curva
fig = go.Figure()
for k, a_ in [(3, 0.35), (2, 0.65), (1, 1.0)]:
    fig.add_trace(go.Scatter(x=cv.meses_alarme_ativos_pct, y=100 * cv[f"canc_k{k}"] / N_CANC, mode="lines", name=f"Cancelados detectados em −{k}",
                             line=dict(color=rgba(CORES["Cancelou"], a_), width=2.5, shape="hv"),
                             customdata=np.c_[cv.corte, cv[f"canc_k{k}"], cv.ativos_hoje],
                             hovertemplate=f"Corte p ≥ %{{customdata[0]:.2f}}<br>Detecta %{{customdata[1]}} de 22 em −{k}"
                                           "<br>Meses em alarme nos ativos: %{x:.1f}%<br>Ativos em alarme hoje: %{customdata[2]}<extra></extra>"))
for m, simb in [("regra", "x"), ("so_uso", "diamond")]:
    r = BL[m]
    for k, a_ in [(3, 0.35), (2, 0.65), (1, 1.0)]:
        fig.add_trace(go.Scatter(x=[r["meses_alarme_ativos_pct"]], y=[100 * r[f"canc_k{k}"] / N_CANC], mode="markers",
                                 name=NOME[m], legendgroup=m, showlegend=k == 1,
                                 marker=dict(symbol=simb, size=12, color=rgba(COR_METODO[m], a_), line=dict(color=COR_METODO[m], width=1.5)),
                                 hovertemplate=f"<b>{NOME[m]}</b><br>Detecta {int(r[f'canc_k{k}'])} de 22 em −{k}"
                                               f"<br>Meses em alarme nos ativos: %{{x:.1f}}%<extra></extra>"))
for nome, c, cor in [("alto", CUT["alto"], CORES["Cancelou"]), ("atenção", CUT["atencao"], AMBAR)]:
    q = cv.loc[np.isclose(cv.corte, c)].iloc[0]
    fig.add_vline(x=q.meses_alarme_ativos_pct, line=dict(color=cor, dash="dash", width=1.5))
    fig.add_annotation(x=q.meses_alarme_ativos_pct, y=4, xanchor="left", yanchor="bottom", showarrow=False, font=dict(size=12, color=cor),
                       text=f"<b>corte {nome}: p ≥ {br(c)}</b><br>{int(q.canc_k1)}/{int(q.canc_k2)}/{int(q.canc_k3)} cancelados em −1/−2/−3"
                            f"<br>{int(q.ativos_hoje)} ativos em alarme hoje<br>precisão nos meses marcados: {pct(q.precisao_linhas, 0)}")
fig.update_xaxes(title_text="Alarme falso: % dos meses (últimos 12) em que um ativo ficou em alarme", ticksuffix="%", range=[-0.5, 32])
fig.update_yaxes(title_text="% dos 22 cancelados detectados", ticksuffix="%", range=[0, 104])
fig.update_layout(height=560, legend=dict(y=1.06),
                  title="Detecção × alarme falso ao mover o corte de probabilidade: onde ficam as faixas"
                        + sub(f"Alto = menor corte com ≥ 50% de precisão nos meses-cliente marcados · Atenção = maior corte que pega em −2 e −3 "
                              f"ao menos tantos cancelados quanto a regra · Youden (referência) = {br(CUT['youden'])} · sem limite de capacidade da equipe"))
fig.show()

## 4. Fila de atendimento (jun/2026)

In [5]:
def evid_txt(c, n=3):
    e = E[(E.cliente_id == c) & E.dispara].head(n)
    return "<br>".join(f"• {t}: {dt}" for t, dt in zip(e.titulo, e.detalhe)) or "—"

FILL = {"alto": "#fbeaea", "atencao": "#fdf3e1", "baixo": "white"}
cols = {"Prioridade": F.prioridade.astype(str), "Cliente": F.index, "Segmento · plano": d.clientes.set_index("cliente_id").loc[F.index].apply(lambda r: f"{r.segmento} · {r.plano}", axis=1),
        "Valor mensal": F.valor_mensal.map(fmt_brl), "Probabilidade (3 meses)": F.probabilidade.map(pct),
        "Perda anual esperada": F.perda_anual_esperada.map(fmt_brl), "Faixa": F.faixa_risco.map(NOME_FAIXA),
        "Principais evidências": [evid_txt(c) for c in F.index], "Ação sugerida": F.acao_sugerida}
fig = go.Figure(go.Table(columnwidth=[0.7, 0.7, 1.4, 1, 1, 1.1, 0.8, 4.2, 2.4],
                         header=dict(values=[f"<b>{c}</b>" for c in cols], fill_color="#f3f2ee", align="left", font_size=12),
                         cells=dict(values=list(cols.values()), align="left", font_size=11, height=26,
                                    fill_color=[[FILL[f] for f in F.faixa_risco]] * len(cols))))
nf = F.faixa_risco.value_counts()
em = F[F.faixa_risco != "baixo"]
fig.update_layout(height=2200, margin=dict(l=10, r=10, t=110, b=10),
                  title=f"Fila dos {len(F)} ativos por perda anual esperada: {nf.get('alto', 0)} em risco alto e {nf.get('atencao', 0)} em atenção "
                        f"({fmt_brl(em.valor_mensal.sum())}/mês)"
                        + sub("Prioridade = probabilidade de cancelar nos próximos 3 meses × valor mensal × 12 · "
                              "evidências = variáveis que empurram o risco para cima E passam do limiar do notebook 02"))
fig.show()

## 5. Risco × valor em jogo

In [6]:
fig = go.Figure()
vx = np.geomspace(2000, 45000, 50)
for perda in (2_000, 10_000, 50_000):
    fig.add_trace(go.Scatter(x=vx, y=perda / (12 * vx), mode="lines", line=dict(color="#d6d4ce", dash="dot", width=1),
                             hoverinfo="skip", showlegend=False))
    fig.add_annotation(x=np.log10(40000), y=np.log10(perda / (12 * 40000)), text=f"perda esperada {fmt_brl(perda)}/ano",
                       showarrow=False, font=dict(size=10, color=CINZA), yanchor="bottom", xanchor="right")
for fx in ("baixo", "atencao", "alto"):
    s = F[F.faixa_risco == fx]
    rot = s.index.where((s.faixa_risco != "baixo") | (s.prioridade <= 10), "")
    fig.add_trace(go.Scatter(x=s.valor_mensal, y=s.probabilidade, mode="markers+text", name=f"{NOME_FAIXA[fx]} ({len(s)})", text=rot,
                             textposition="top center", textfont=dict(size=11, color=COR_FAIXA[fx]),
                             marker=dict(color=rgba(COR_FAIXA[fx], 0.8), size=np.clip(np.sqrt(s.perda_anual_esperada) / 8, 7, 34),
                                         line=dict(color="white", width=1.5)),
                             customdata=np.c_[s.index, s.prioridade, s.perda_anual_esperada.map(fmt_brl), [evid_txt(c, 4) for c in s.index], s.acao_sugerida],
                             hovertemplate="<b>%{customdata[0]}</b> · prioridade %{customdata[1]}<br>R$ %{x:,.0f}/mês · probabilidade %{y:.1%}"
                                           "<br>Perda anual esperada: %{customdata[2]}<br>%{customdata[3]}<br><i>%{customdata[4]}</i><extra></extra>"))
for nome, c, cor in [("alto", CUT["alto"], CORES["Cancelou"]), ("atenção", CUT["atencao"], AMBAR)]:
    fig.add_hline(y=c, line=dict(color=cor, dash="dash", width=1.2))
    fig.add_annotation(x=np.log10(2100), y=np.log10(c), text=f"corte {nome} ({pct(c, 0)})", showarrow=False, xanchor="left", yanchor="bottom",
                       font=dict(size=11, color=cor))
fig.update_xaxes(type="log", title_text="Valor mensal do contrato (R$, escala log)", tickprefix="R$ ", tickformat=",.0f", range=[np.log10(2000), np.log10(45000)])
fig.update_yaxes(type="log", title_text="Probabilidade de cancelar nos próximos 3 meses (escala log)", tickformat=".1%", range=[np.log10(0.0008), np.log10(0.8)])
fig.update_layout(height=640, legend=dict(y=1.04),
                  title=f"Risco × valor: {F.index[0]} lidera a fila (probabilidade {pct(F.probabilidade.iloc[0], 0)} em "
                        f"{fmt_brl(F.valor_mensal.iloc[0])}/mês); contratos grandes sobem mesmo com risco menor"
                        + sub("Bolha = perda anual esperada · linhas pontilhadas = mesma perda esperada · rótulo: faixa alto/atenção e 10 primeiros da fila"))
fig.show()

## 6. Por que os 3 primeiros da fila

In [7]:
TOP3 = F.index[:3].tolist()
C3 = contribuicoes(M, F.loc[TOP3])
b0 = M.intercepto()
sig = lambda z: 1 / (1 + np.exp(-z))
fig = make_subplots(rows=3, cols=1, vertical_spacing=0.13,
                    subplot_titles=[f"<b>{c}</b> · prioridade {F.at[c, 'prioridade']} · {fmt_brl(F.at[c, 'valor_mensal'])}/mês · "
                                    f"probabilidade {pct(F.at[c, 'probabilidade'])} · faixa {NOME_FAIXA[F.at[c, 'faixa_risco']].lower()}" for c in TOP3])
for i, c in enumerate(TOP3, start=1):
    ct = C3.loc[c].reindex(C3.loc[c].abs().sort_values(ascending=False).index)
    ec = E[E.cliente_id == c].set_index("variavel")
    marca = ["▲ " if ec.at[f, "dispara"] else "" for f in ct.index]
    xs = ["Cliente médio"] + [f"{m}{ROT[f]}" for f, m in zip(ct.index, marca)] + ["Este cliente"]
    fig.add_trace(go.Waterfall(x=xs, y=[b0, *ct.values, 0], measure=["absolute"] + ["relative"] * len(ct) + ["total"],
                               increasing=dict(marker=dict(color=CORES["Cancelou"])), decreasing=dict(marker=dict(color=CORES["Ativo"])),
                               totals=dict(marker=dict(color=TINTA)), connector=dict(line=dict(color="#c9c7c0", width=1)),
                               text=[br(b0)] + [("+" if v > 0 else "") + br(v) for v in ct.values] + [br(b0 + ct.sum())], textposition="outside",
                               customdata=[f"Probabilidade do cliente médio: {pct(sig(b0))}"]
                                          + [f"{ec.at[f, 'titulo']}<br>{ec.at[f, 'detalhe']}" for f in ct.index]
                                          + [f"Probabilidade: {pct(sig(b0 + ct.sum()))}"],
                               hovertemplate="<b>%{x}</b><br>%{customdata}<br>Log-odds: %{y:+.2f}<extra></extra>", showlegend=False), row=i, col=1)
    fig.update_yaxes(title_text="log-odds", row=i, col=1)
fig.update_xaxes(tickangle=-25, tickfont_size=11)
fig.update_layout(height=1250, margin=dict(t=140, b=80),
                  title="De onde vem o risco dos 3 primeiros: contribuição de cada variável sobre o cliente médio"
                        + sub(f"Contribuição = coeficiente × valor padronizado (log-odds) · vermelho aumenta, azul reduz · "
                              f"▲ = evidência (passa do limiar do notebook 02) · cliente médio = {pct(sig(b0))}"))
fig.show()